## Explanation for Task 1

In this task, I collected metadata for 400 unique arxiv papers from the year 2025.

First, I created a function to randomly generate arxiv IDs in the 2025 format. Since arxiv IDs use the format `YYMM.NNNNN`, the generated IDs start with `25`, representing the year 2025. The month is randomly selected from 1 to 12, and the paper number is also randomly generated.

Then, I used the `arxiv` Python package to search for papers matching these randomly generated arxiv IDs. The code collects metadata only for valid papers whose IDs start with `25` and avoids duplicate papers by storing collected IDs in a set.

For each paper, I collected important metadata such as the arXiv ID, title, authors, number of authors, summary, published date, updated date, primary category, all categories, PDF URL, and entry URL.

After collecting 400 unique papers, I converted the data into a pandas DataFrame.

Finally, I saved the collected metadata into a CSV file named `arxiv_papers_2025.csv`, which will be used for the next tasks in the project.

In [36]:
# Task 1 - Arxiv Paper Metadata Collection
!pip install arxiv

In [37]:
import arxiv
import random
import time
import pandas as pd

In [38]:
# Step 1: Generating random 400 arxiv ID
# Function to generate a random arxiv ID from the year 2025

def generate_random_2025_arxiv_id():
    """
    Generates a random arXiv ID for the year 2025.
    Example: 2501.00001
    """
    month = random.randint(1, 12)
    paper_number = random.randint(1, 99999)

    arxiv_id = f"25{month:02d}.{paper_number:05d}"
    return arxiv_id

In [39]:
# Step 2: Collecting the metadata from the research paper
# Total number of papers required is 400
TARGET_PAPERS = 400

# Storing collected paper records
papers_data = []

# Storing unique arXiv IDs in a set to avoid duplicates
collected_ids = set()

# Creating an arxiv client
client = arxiv.Client(
    page_size=50,
    delay_seconds=3,
    num_retries=3
)

BATCH_SIZE = 50

print("Starting the metadata collection")

while len(papers_data) < TARGET_PAPERS:

    # Generating a batch of random candidate IDs
    candidate_ids = []

    while len(candidate_ids) < BATCH_SIZE:
        random_id = generate_random_2025_arxiv_id()

        if random_id not in collected_ids:
            candidate_ids.append(random_id)

    # Searching arxiv using the generated IDs
    search = arxiv.Search(id_list=candidate_ids)

    try:
        results = list(client.results(search))

        for paper in results:
            # Extracting clean arxiv ID without version number
            clean_id = paper.get_short_id().split("v")[0]

            # Making sure if the paper belongs to 2025 and it is unique
            if clean_id.startswith("25") and clean_id not in collected_ids:

                paper_record = {
                    "arxiv_id": clean_id,
                    "title": paper.title,
                    "authors": [author.name for author in paper.authors],
                    "author_count": len(paper.authors),
                    "summary": paper.summary,
                    "published": paper.published,
                    "updated": paper.updated,
                    "primary_category": paper.primary_category,
                    "categories": paper.categories,
                    "pdf_url": paper.pdf_url,
                    "entry_id": paper.entry_id
                }

                papers_data.append(paper_record)
                collected_ids.add(clean_id)

                print(f"Collected {len(papers_data)} / {TARGET_PAPERS}: {clean_id}")

                if len(papers_data) >= TARGET_PAPERS:
                    break

    except Exception as e:
        print("Error occurred:", e)
        time.sleep(5)

print("Metadata collection completed.")

Starting the metadata collection
Collected 1 / 400: 2501.16349
Collected 2 / 400: 2501.04834
Collected 3 / 400: 2507.08695
Collected 4 / 400: 2503.23527
Collected 5 / 400: 2512.03699
Collected 6 / 400: 2509.11217
Collected 7 / 400: 2505.22406
Collected 8 / 400: 2511.20528
Collected 9 / 400: 2512.00480
Collected 10 / 400: 2510.27200
Collected 11 / 400: 2511.10391
Collected 12 / 400: 2511.01031
Collected 13 / 400: 2503.13908
Collected 14 / 400: 2508.18980
Collected 15 / 400: 2512.05423
Collected 16 / 400: 2507.04500
Collected 17 / 400: 2507.07282
Collected 18 / 400: 2508.08136
Collected 19 / 400: 2508.15824
Collected 20 / 400: 2510.09633
Collected 21 / 400: 2511.17565
Collected 22 / 400: 2512.06220
Collected 23 / 400: 2512.03132
Collected 24 / 400: 2505.20780
Collected 25 / 400: 2503.12292
Collected 26 / 400: 2510.15012
Collected 27 / 400: 2509.13612
Collected 28 / 400: 2511.04976
Collected 29 / 400: 2511.09174
Collected 30 / 400: 2507.05152
Collected 31 / 400: 2511.13213
Collected 32 / 

In [40]:
# Step 3: Converting to DataFrame

df = pd.DataFrame(papers_data)

# Converting the published and updated to date only
df["published"] = pd.to_datetime(df["published"]).dt.date
df["updated"] = pd.to_datetime(df["updated"]).dt.date

# Converting authors list to comma separated string
df["authors"] = df["authors"].apply(lambda x: ", ".join(x))

# Checking the earliest and latest research paper in 2025
df["arxiv_month"] = df["arxiv_id"].astype(str).apply(lambda x: int(x[2:4]))
earliest = df["arxiv_id"].astype(str).apply(lambda x: int(x[2:4])).min()
latest   = df["arxiv_id"].astype(str).apply(lambda x: int(x[2:4])).max()

print(f"Shape: {df.shape}")
print(f"\nDate range of papers:")
print(f"  Earliest month: 2025-{earliest:02d}")
print(f"  Latest month:   2025-{latest:02d}")
print(f"\nMonth distribution:")

# Checking how many papers are there in each month
df["arxiv_year_month"] = df["arxiv_id"].astype(str).apply(lambda x: f"2025-{x[2:4]}")
month_counts = df["arxiv_year_month"].value_counts().sort_index()
print("Papers per month (from arxiv_id):")
print(month_counts)

print(f"\nFirst 5 rows:")
df.head()

Shape: (400, 12)

Date range of papers:
  Earliest month: 2025-01
  Latest month:   2025-12

Month distribution:
Papers per month (from arxiv_id):
arxiv_year_month
2025-01    22
2025-02    26
2025-03    30
2025-04    42
2025-05    39
2025-06    33
2025-07    28
2025-08    27
2025-09    34
2025-10    44
2025-11    33
2025-12    42
Name: count, dtype: int64

First 5 rows:


,arxiv_id,title,authors,author_count,summary,published,updated,primary_category,categories,pdf_url,entry_id,arxiv_month,arxiv_year_month
0,2501.16349,Risk-Informed Diffusion Transformer for Long-T...,"Junlan Chen, Pei Liu, Zihao Zhang, Hongyi Zhao...",6,Trajectory prediction methods have been widely...,2025-01-18,2025-05-28,cs.LG,"[cs.LG, cs.AI]",https://arxiv.org/pdf/2501.16349v2,http://arxiv.org/abs/2501.16349v2,1,2025-01
1,2501.04834,Detectability of Emission from Exoplanet Outfl...,"Riley Rosener, Michael Zhang, Jacob L. Bean",3,Photoevaporation in exoplanet atmospheres is t...,2025-01-08,2025-01-10,astro-ph.EP,"[astro-ph.EP, astro-ph.IM]",https://arxiv.org/pdf/2501.04834v2,http://arxiv.org/abs/2501.04834v2,1,2025-01
2,2507.08695,The optimal Padé polynomial for reconstruction...,"Bo Yu, Wenhu Liu, XiaoFeng Yang, Tong-Jie zhan...",5,The cosmography known as the Padé polynomials ...,2025-07-11,2025-07-11,astro-ph.CO,[astro-ph.CO],https://arxiv.org/pdf/2507.08695v1,http://arxiv.org/abs/2507.08695v1,7,2025-07
3,2503.23527,Convergent Power Series for Anharmonic Chain w...,"Pedro L. Garrido, Tomasz Komorowski, Joel L. L...",4,We study the propagation of energy in one-dime...,2025-03-30,2026-01-10,math-ph,[math-ph],https://arxiv.org/pdf/2503.23527v5,http://arxiv.org/abs/2503.23527v5,3,2025-03
4,2512.03699,Transitivity and an abelian Livsic theorem for...,"Mark Pollicott, Richard Sharp",2,We show that the abelian Livšic theorem recent...,2025-12-03,2025-12-03,math.DS,[math.DS],https://arxiv.org/pdf/2512.03699v1,http://arxiv.org/abs/2512.03699v1,12,2025-12


In [41]:
# Step 4: Saving the data in a CSV file

df.to_csv("arxiv_papers_2025.csv", index=False)
print("Data saved to arxiv_papers_2025.csv")
print(f"Total records saved: {len(df)}")

Data saved to arxiv_papers_2025.csv
Total records saved: 400
